
## Patch Instruction for `AutoPipeline` Before Running MASE Pipeline

To prevent the `TypeError: object of type 'NoneType' has no len()` error, follow these steps:

1. **Open the file**: ./mase/src/chop/pipelines/auto_pipeline.py

2. **Locate line 26** inside the `AutoPipeline.__init__` method.

3. **Replace the existing line** with:

```python
self.pass_outputs = [{}] * len(self.pass_groups)

In [ ]:
# !git clone --branch jtan/dev_physical https://github.com/tanjun8802/Mase_EDGE.git 
# %cd /content
# !rm -rf Mase_EDGE/mase
# %cd Mase_EDGE
!git submodule update --init --recursive
%cd mase/
%pip install -e ".[executorch]"

In [ ]:
%cd /content/Mase_EDGE
!wget http://cs231n.stanford.edu/tiny-imagenet-200.zip
!unzip tiny-imagenet-200.zip

In [ ]:
import sys
from pathlib import Path

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import torchvision.transforms as T
import torchvision.datasets as datasets
import torchvision.models as models

_cwd = Path.cwd().resolve()
_REPO = _cwd
for c in (_cwd, _cwd.parent, _cwd.parent.parent):
    if (c / "tiny_imagenet_lib").is_dir():
        _REPO = c
        break
if str(_REPO) not in sys.path:
    sys.path.insert(0, str(_REPO))

from tiny_imagenet_lib import evaluate, RESNET_QUANTIZATION_CONFIG

device = "cuda" if torch.cuda.is_available() else "cpu"

transform = T.Compose([
    T.Resize(224),
    T.ToTensor(),
    T.Normalize([0.485, 0.456, 0.406],
                [0.229, 0.224, 0.225])
])

train_dataset = datasets.ImageFolder(root="./tiny-imagenet-200/train", transform=transform)
val_dataset   = datasets.ImageFolder(root="./tiny-imagenet-200/val", transform=transform)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=64, shuffle=False)

model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
model.fc = nn.Linear(model.fc.in_features, 200)
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)
epochs = 1

for epoch in range(epochs):
    model.train()
    running_loss = 0
    for imgs, labels in train_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        loss = criterion(model(imgs), labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    print(f"FP32 Epoch {epoch+1}/{epochs}, Loss: {running_loss/len(train_loader):.4f}")

acc_fp32 = evaluate(model, val_loader, device=device)
print(f"✅ FP32 Accuracy: {acc_fp32:.2f}%")

from chop import MaseGraph
import chop.passes as passes

example_inputs = torch.randn(1, 3, 224, 224).to(device)
dummy_inputs = {"x": example_inputs}

mg = MaseGraph(model)
mg, _ = passes.init_metadata_analysis_pass(mg)
mg, _ = passes.add_common_metadata_analysis_pass(mg, pass_args={"dummy_in": dummy_inputs})

quantization_config = RESNET_QUANTIZATION_CONFIG

mg, _ = passes.quantize_transform_pass(mg, pass_args=quantization_config)
print("✅ PTQ applied to ResNet18.")

ptq_model = torch.fx.GraphModule(mg.model, mg.fx_graph).to(device)

acc_ptq = evaluate(ptq_model, val_loader, device=device)
print(f"✅ PTQ Accuracy: {acc_ptq:.2f}%")

qat_model = model
optimizer = optim.Adam(qat_model.parameters(), lr=1e-4)
epochs_qat = 1

for epoch in range(epochs_qat):
    qat_model.train()
    running_loss = 0
    for imgs, labels in train_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        loss = criterion(qat_model(imgs), labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    print(f"QAT Epoch {epoch+1}/{epochs_qat}, Loss: {running_loss/len(train_loader):.4f}")

acc_qat = evaluate(qat_model, val_loader, device=device)
print(f"✅ QAT Accuracy: {acc_qat:.2f}%")

output_dir = Path("./mase_output")
output_dir.mkdir(exist_ok=True)

qat_model.eval()
torch.onnx.export(
    qat_model,
    example_inputs,
    str(output_dir / "resnet18_qat_fp32.onnx"),
    opset_version=17,
    input_names=["x"],
    output_names=["output"],
    dynamic_axes={"x": {0: "batch_size"}, "output": {0: "batch_size"}}
)
print(f"✅ ONNX FP32 QAT model exported to {output_dir / 'resnet18_qat_fp32.onnx'}")

CHECKPOINTS_DIR = _REPO / "checkpoints"
CHECKPOINTS_DIR.mkdir(parents=True, exist_ok=True)
ckpt_path = CHECKPOINTS_DIR / "resnet18_qat_fp32.pt"
torch.save({"state_dict": qat_model.state_dict(), "num_classes": 200}, ckpt_path)
print(f"✅ PyTorch weights saved to {ckpt_path}")
